# Day 4: Melodic Similarity
**Date:** Wednesday 24 June 2026

**Conceptual frame:** Every similarity metric encodes a theory of what matters about melody.
Choosing a metric is making a musicological argument.


```{admonition} Conceptual check — before you code
:class: tip

Answer the self-assessment questions for Day 4 before running the cells below.
Questions open in a new tab — come back here when you're done.

**[→ Open Day 4 Quiz](../quizpages/day4_quiz.md)**
```


---
## Part 1: Setup


In [ ]:
import requests, zipfile
from pathlib import Path
from collections import Counter
from itertools import islice
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from music21 import converter, note, interval
sns.set_theme(style='whitegrid',font_scale=1.1)
plt.rcParams['figure.figsize']=(10,4)
print('OK')

In [ ]:
CORPUS_DIR=Path('beregovski_corpus');KERN_DIR=CORPUS_DIR/'kern'
if not(KERN_DIR.exists() and list(KERN_DIR.glob('*.krn'))):
    CORPUS_DIR.mkdir(exist_ok=True)
    r=requests.get('https://github.com/shanahdt/mode_in_klezmer/archive/refs/heads/main.zip')
    zp=CORPUS_DIR/'repo.zip';zp.write_bytes(r.content)
    import shutil
    with zipfile.ZipFile(zp) as z:z.extractall(CORPUS_DIR)
    src=list(CORPUS_DIR.glob('mode_in_klezmer-*/kern'))
    if src:
        if KERN_DIR.exists():shutil.rmtree(KERN_DIR)
        shutil.copytree(src[0],KERN_DIR);zp.unlink()
print(f'{len(list(KERN_DIR.glob("*.krn")))} kern files ready')

In [ ]:
def load_corpus(kern_dir=KERN_DIR, verbose=True):
    pc2d={7:1,9:2,11:3,0:4,2:5,4:6,6:7,8:2,10:3,1:4,3:5,5:6}
    records,sdict={},{}
    for i,f in enumerate(sorted(Path(kern_dir).glob('*.krn'))):
        if verbose and i%50==0: print(f'  {i+1}...')
        try:
            s=converter.parse(str(f));ns=[n for n in s.flat.notes if isinstance(n,note.Note)]
            pcs=[n.pitch.pitchClass for n in ns]
            records[f.stem]={'tune_id':f.stem,'n_notes':len(ns),
                'pitches':[n.nameWithOctave for n in ns],'pitch_classes':pcs,
                'scale_degrees':[pc2d.get(p,0) for p in pcs],
                'intervals':[interval.Interval(ns[j],ns[j+1]).semitones for j in range(len(ns)-1)]}
            sdict[f.stem]=s
        except: pass
    if verbose: print(f'Loaded {len(records)} tunes.')
    return pd.DataFrame(records.values()),sdict

def get_ngrams(seq,n): return list(zip(*[islice(seq,i,None) for i in range(n)]))
print('Helpers ready.')

In [ ]:
df,streams=load_corpus()
try:
    meta=pd.read_csv('https://raw.githubusercontent.com/shanahdt/mode_in_klezmer/main/metadata.csv')
    df=df.merge(meta,on='tune_id',how='left')
    print(f'{len(df)} tunes loaded with metadata')
except Exception as e:
    print(f'Metadata unavailable ({e})')

---
## Part 2: Edit Distance

| Representation | Preserves | Ignores |
|---|---|---|
| Absolute pitches | Exact notes | Transposition |
| Scale degrees | Tonal function | Absolute pitch |
| Intervals | Motion patterns | Starting pitch |
| Contour | Shape (up/down/same) | Interval size |


In [ ]:
def edit_distance(s1,s2):
    m,n=len(s1),len(s2)
    dp=[[0]*(n+1) for _ in range(m+1)]
    for i in range(m+1): dp[i][0]=i
    for j in range(n+1): dp[0][j]=j
    for i in range(1,m+1):
        for j in range(1,n+1):
            dp[i][j]=dp[i-1][j-1] if s1[i-1]==s2[j-1] else 1+min(dp[i-1][j],dp[i][j-1],dp[i-1][j-1])
    return dp[m][n]

def norm_edit(s1,s2): return edit_distance(s1,s2)/max(len(s1),len(s2),1)

def jaccard_ngram(s1,s2,n=3):
    a,b=set(get_ngrams(s1,n)),set(get_ngrams(s2,n))
    return len(a&b)/len(a|b) if a|b else 1.0

def melodic_contour(pitches):
    from music21.pitch import Pitch
    ps=[Pitch(p).ps for p in pitches]
    return [1 if ps[i]>ps[i-1] else (-1 if ps[i]<ps[i-1] else 0) for i in range(1,len(ps))]

def contour_sim(p1,p2):
    c1,c2=melodic_contour(p1),melodic_contour(p2)
    n=min(len(c1),len(c2))
    return sum(1 for a,b in zip(c1[:n],c2[:n]) if a==b)/n if n else 0.0

# Quick test
t1=df.iloc[0];t2=df.iloc[1]
print(f'Tunes: {t1.tune_id} vs {t2.tune_id}')
print(f'Edit (norm):  {norm_edit(t1.scale_degrees,t2.scale_degrees):.3f}')
print(f'Jaccard:      {jaccard_ngram(t1.scale_degrees,t2.scale_degrees):.3f}')
print(f'Contour:      {contour_sim(t1.pitches,t2.pitches):.3f}')

---
## Part 3: Pairwise Similarity Matrix


In [ ]:
SUBSET_MODE='freygish';N=20
subset=df[df['mode']==SUBSET_MODE].head(N) if 'mode' in df.columns else df.head(N)
tids=subset['tune_id'].tolist();n=len(tids)

edit_mat=np.zeros((n,n));jac_mat=np.zeros((n,n));con_mat=np.zeros((n,n))
for i,t1 in enumerate(tids):
    r1=df[df.tune_id==t1].iloc[0]
    for j,t2 in enumerate(tids):
        r2=df[df.tune_id==t2].iloc[0]
        edit_mat[i,j]=norm_edit(r1.scale_degrees,r2.scale_degrees)
        jac_mat[i,j]=1-jaccard_ngram(r1.scale_degrees,r2.scale_degrees)
        con_mat[i,j]=1-contour_sim(r1.pitches,r2.pitches)

fig,axes=plt.subplots(1,3,figsize=(15,4))
for ax,mat,title in zip(axes,[edit_mat,jac_mat,con_mat],
                         ['Edit distance','Jaccard (trigram)','Contour']):
    sns.heatmap(mat,ax=ax,cmap='YlOrRd',xticklabels=False,yticklabels=False)
    ax.set_title(title)
fig.suptitle(f'Pairwise similarity — {N} {SUBSET_MODE} tunes (darker=more similar)')
plt.tight_layout();plt.show()

In [ ]:
diffs=[(abs(edit_mat[i,j]-jac_mat[i,j]),tids[i],tids[j],edit_mat[i,j],jac_mat[i,j])
       for i in range(n) for j in range(i+1,n)]
diffs.sort(reverse=True)
print('Top 5 metric disagreements (edit vs Jaccard):')
print(f'{'Tune A':25s} {'Tune B':25s} {'Edit':>8s} {'Jaccard':>8s} {'Diff':>7s}')
for diff,t1,t2,ed,jac in diffs[:5]:
    print(f'{t1:25s} {t2:25s} {ed:8.3f} {jac:8.3f} {diff:7.3f}')

---
## Day 4 Exercise: Metric Disagreement

```{admonition} Exercise
Find one tune pair where edit distance and n-gram Jaccard give substantially different scores.
Write 150–200 words: which metric's judgment is more musically defensible for this specific pair,
and what does your argument imply about what 'similarity' means in this repertoire?
```


In [ ]:
TUNE_A=tids[0];TUNE_B=tids[1]  # <-- change to your disagreement pair
ra=df[df.tune_id==TUNE_A].iloc[0];rb=df[df.tune_id==TUNE_B].iloc[0]
print(f'A: {ra.scale_degrees[:25]}')
print(f'B: {rb.scale_degrees[:25]}')
print(f'Edit: {norm_edit(ra.scale_degrees,rb.scale_degrees):.3f}')
print(f'Jaccard: {jaccard_ngram(ra.scale_degrees,rb.scale_degrees):.3f}')
print(f'Contour: {contour_sim(ra.pitches,rb.pitches):.3f}')

### Your argument

*(150–200 words)*


---
## Project Log — Entry 4

> *Most similar pairs under edit distance: [X]. Under Jaccard: [Y].*  
> *Most interesting disagreement: [Z], because [metric A] is sensitive to [feature] while [metric B] is not.*  
> *A musician would say these tunes are [similar/different] because...*

*(100–150 words)*
